# Transfermarkt data cleaning part 3:

## Introduction
This notebook is about the creation of the team's match importance and team pressure related features.Also there could be made some validations and final data preparations!

### About the features that will be created:
The features will be related to the matches importance and team presure.Not all of the matches have equal meaning to the teams, as ones are more **important** and ones are more **neutral**.By this I don't mean that a specific match is absolutely worthless and unimportant, but I want to specify that there are some more specific situations and cases where certain matches are more important than others.And this is the exact thing that this notebook will try to capture.


### Limitations:
In order these features to be created, we need to take into accout a lot of **specifications** about the league, the teams, the positions, the competitions, rounds etc!However, all of these **specifications** are static and known to be true, so we will not have problems with them - for example it is known that in La Liga, a team is **qualifying** for the **Champions League** if he is **top 4/5** in the competition.However, the problem occurs when we try to capture the team specific match importance - What I mean by this is that every team **has different priorities** based on his state and condition in time of the season and we cannot really define the real match importance for every single team!

Something else which is extremely important is the **time factor**.How the time factor influence the match importance and team pressure: Imagine a team at the beggining of the season - at the start of the season the state of the teams is too unsettled and it is still not clear who the **favorites** are and the ones who differ.Now imagine the teams at the end of the competition.Every team is fighting for something, either for the title or to not be relegated!At that time of the season, when everything is known and each of the teams has its own specific aim, the matches importances can be classified much more easlily!

### Main Aim
As I said, every team has different priorities and different aims, depending on the period of the season and the team's condition(state)!This is something absolutely logical and understandable - if everything was settled for each team and every team intention is known, it wouldn't have been so interesting right(:!However, our aim, when creating the features, is to capture the different states of the teams at the different periods of the season.What's more I will try to add a little bit more info about what is happening around the team at the time of the matches - with this I mean that I will try to see the bigger picture, not just the matches themselves, because most of the times, the **real truth** is hidden in the **details** and not in the so **obvious things**!

### Process of work:
Now lets explain some more details about the implementation of the features and how they will be created! \
Here is the exact way that I will divide the implementation of the features:

**1. LEAGUE STANDING PRESSURE**
   Points, goal difference, points gaps to title / top-4 / top-6 / relegation. \
   Pressure flags that activate based on gap thresholds and season progress.
 
**2. SEASON PROGRESS**
   Round-based progress factor that amplifies pressure signals as the season matures. \
   Early-season matches are down-weighted; late-season matches are amplified.This is very important because, as I said the time factor plays a huge role in the importance of the matches, because as the season grow, the team's places and performance become more clear and the matches become more important because at the end every team is fighting for something!
 
**3. DIRECT CONFRONTATION**
   Flags and gap features when the two teams are in the same race
   (title, European spots, relegation battle).
 
**4. EXTERNAL COMPETITION PRESSURE**
   Priority of the team's next non-league match, days until it arrives, \
   and whether an important cup / European match is coming.
 
**5. FIXTURE CONGESTION**
   Matches played in the last 7 / 14 days across all competitions. \
   Rotation risk score combining congestion and upcoming match priority.
 
**6. COMPOSITE MATCH IMPORTANCE**
   A single normalised score [0, 1] summarising how important this specific \
   La Liga match is for each team, combining all of the above.


### About the data that will be used:
For the features to be created, the notebook will work with the `matches_interim_df` dataset, which contains team matches data for all of the most important competitons in europe!

### About the feature's target:
As the object of the project is the **La Liga matches**, the features will be created only for the La Liga matches and for no other competition!This is simply because the models that will be created will try to predict the La Liga matches!Of course this does not mean that the other competiions will not be used - they are the most important source of information that we need to create the features.

#### Possible features to be created(These are just indicative features):
In order to speak more specifically and understand what this notebook will try to implement here are some examples of what can identify the **match importance** and the **team pressure**(as features)!

##### Match Importance:
- distance_to_title
- distance_to_top4
- distance_to_relegation - relegation means that a team will be dropped into the lowest league division!
- title_race_flag - if the team is fighting for the title of the league
- relegation_battle_flag - when two teams are fighting to not be dropped off from the division!
- europe_race_flag - if the team is participating in european competitions(like Champions league, League Europe)

##### External competition and team pressure factors:
- days_until_next_match
- next_match_priority
- important_match_ahead
- fixture_congestion_score
- matches_last_14_days
- rotation_risk

Having said this, I think that we are ready to start!
### Getting started

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re
from pathlib import Path
import unicodedata

from football_betting_analysis.config import START_SEASON, END_SEASON, TRANSFERMARKT_DATA_INTERIM_PATH

from football_betting_analysis.data.data_cleaning import optimize_dataframe_memory
from football_betting_analysis.data.save_data_into_file import save_data

from football_betting_analysis.features.features_transformation import transform_match_round
from football_betting_analysis.data.team_match_validation import validate_team_matches
from football_betting_analysis.data.teams_names_mapper import build_team_mapping

### Loading the latest version of the `matches_interim_df`:

In [2]:
matches_interim_df = pd.read_parquet('../../data/interim/transfermarkt_data/interim_matches_v2.parquet')

In [3]:
matches_interim_df.columns

Index(['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id',
       'away_club_id', 'home_club_goals', 'away_club_goals',
       'home_club_position', 'away_club_position', 'home_club_manager_name',
       'away_club_manager_name', 'stadium', 'attendance', 'referee',
       'home_club_formation', 'away_club_formation', 'home_club_name',
       'away_club_name', 'competition_type', 'home_missing_players_count',
       'home_missing_key_players_count', 'home_missing_star_players_count',
       'home_missing_importance_sum', 'home_missing_defenders',
       'home_missing_midfielders', 'home_missing_forwards',
       'home_starting_goalkeeper_missing', 'home_missing_captain',
       'home_missing_top1_player', 'home_missing_top3_player',
       'home_available_strength', 'home_missing_expected_starter_strength',
       'home_missing_gk_strength', 'home_missing_def_strength',
       'home_missing_mid_strength', 'home_missing_fwd_strength',
       'home_missing_importance_r

This is the last version of the `matches_interim_df` that we have saved in the **Transfermarkt cleaning part 2 notebook**!

Now what I will do is that I will define some of the **constants** that will be used in the feature's implementation.The **constants/thresholds** are very important because they define how the features will be created and what meaning will they have.

First lets define the competitions that we will be working with:

In [4]:
competitions_df = pd.read_csv('../../data/raw/transfermarkt_data/competitions.csv')

In [5]:
target_comps = matches_interim_df.competition_id.unique().tolist()

In [6]:
competitions_df[competitions_df.competition_id.isin(target_comps)][['competition_id', 'name', 'type']]

,competition_id,name,type
9,CDR,copa-del-rey,domestic_cup
12,CL,uefa-champions-league,international_cup
13,CLQ,uefa-champions-league-qualifying,international_cup
20,ECLQ,uefa-conference-league-qualifiers,other
21,EL,uefa-europa-league,other
22,ELQ,uefa-europa-league-qualifying,other
23,ES1,laliga,domestic_league
24,EURO,uefa-euro,national_team_competition
26,FIWC,world-cup,national_team_competition
35,KLUB,fifa-klub-wm,other


These are all of the competitions that the `matches_interim_df` contains!However, we will not work with all of them simply because not all of them directly affect our target matches(La Liga matches).We should work only with the competitions that are played during the La Liga matches.And these are: **Champions League, League Europe, Conference league, UEFA Super Cup, Copa Del Ray, Spain Super Cup**!All of the other are international competitions that are played separately from the other competitions!

Ok, we have defined the competitions that we will be working to, however, they are not the same!With I want to specify the two exclusions: **EUFA Super Cup and Spain Super Cup** - These are not standard competitions which are played by many teams and last long!These competitions are played only by specific teams which have either won some trophy or ended in top 2 in the league!

1. *EUFA Super Cup*: \
This competitions is played as a final single match between two teams.The one which has won the **Champions League** and the one which have won the **League Europe**!So as you can see, this is not a standard multilayer competition, but a single match between two teams!

2. *Spain supecopa*: \
This competition is played from four teams: The **top two teams** from La Liga and the finalists from **Copa del Rey**!Again, this is not a standard competition, but a mini tournament between four teams!

> #### I want to specify that these two competitions will be processed differently from the other standard competitions and will not have a very big impact over the match-importance features!

As we specify the target competitions and also classified them, lets define the competitions constants: 

In [7]:
# The target competition:
LA_LIGA_ID = "ES1"

# STANDARD: regular fixtures that count toward congestion / rotation pressure
STANDARD_COMPETITION_IDS = {"ES1", "CDR", "CL", "EL", "UCOL"}

# FINAL-ONLY: rare, finalist/winner-only fixtures — excluded from congestion
# load (they don't represent a "normal" extra fixture burden) but tracked
# separately as a prestige / recent-final signal
FINAL_ONLY_COMPETITION_IDS = {"SUC", "USC"}
 
# EXCLUDED: qualifiers and national-team competitions — not club pressure signals
# These list contains qualifications which are played from more lower-ranked coutries 
# who do not get automatic group entry - this is not related to the Speanish La Liga as one of
# the best leagues in Europe!
EXCLUDED_COMPETITION_IDS = {"CLQ", "ECLQ", "ELQ", "EURO", "FIWC", "KLUB"}

# Each season in a football domestic competition(like LaLiga) has 38 rounds with 10 matches in each round!
SEASON_ROUNDS = 38

Now something which is very important is to clasify the competitions by their **priorities**.

The way thay I am going to do that is by following the conventions and standards that have developed in football in the last decade!**[3]**

In [8]:
# Competition priority weights — used for rotation risk / next-match importance.
# Only STANDARD + FINAL_ONLY competitions need a weight; excluded ones are
# filtered out before any priority lookup happens.
COMPETITION_PRIORITY = {
    "CL":   1.00, # Champions League
    "EL":   0.75, # Europa League
    "UCOL": 0.55, # Conference League
    "CDR":  0.60, # Copa del Rey
    "ES1":  0.80, # La Liga
    "USC":  0.65, # UEFA Super Cup (final-only, high prestige when it occurs)
    "SUC":  0.55, # Spanish Supercopa (final-only, moderate prestige)
}

Now the next constants will be about the **pressure flags** that will be created.And these constants will specify the maximum amount of points that a team is allowed to have under the different positions(top1: title, top4: CL, top6: EL, relagation).They will serve as a gap between the team's points and the minumum points needed for the team to reach a specific position in the league such as top1, top4 etc!

I want to specify that these are **just generic constants** which may change depending on **the season and the team**!

In [9]:
# Points thresholds for pressure flags (tunable)
TITLE_RACE_GAP_MAX = 6 # max points gap for a team to get into the title race
TOP4_GAP_MAX = 5 #  max points gap for a team to get into the Champions league race, 
                 #which race is between the top 4/5teams - this depends on the season
TOP6_GAP_MAX = 5 
RELEGATION_GAP_MAX = 5 # max points gapm between the teams points and the 18th team points to get into the relegation race
                       # The last three teams are dropped off from the division and they enter the second division of Spain

# This constant is very important, because it specifies the round above which the pressure activation starts
# and the matches become more important.
# This is made so that to not be put too much presure and importance at the start of the season
# where the team's positions and performanve is not defined yer and the competitions state is unsettled!
PRESSURE_ACTIVATION_ROUND = 10

# The last three teams which are relefated from the first division(La Liga)
RELEGATION_ZONE_SIZE = 3

As I said earlier, there are a lot of specification and things which differ dependeing on the time!I am going to create a function which based on the match season, it returns different European cups places.By places I mean how many team from La Liga are qualified to the Champions League, League Europe and Conference League!

Up until season 2023/2024 the top 4 teams in LaLiga have been qualified for the CL, the 5th and 6th for the LE and the 7th for the COL(conference league).However in season 2024/2025 eufa have expanded the places for CL to 5 and left only 1 for both LE and COL!

So these are thing that should be taken into account when creating the features!

In [10]:
# Champions League spot expansion by season
# Total European places = 7 in both eras; only the CL/EL split changes.
# Pre-2024/25 : 4 CL + 2 EL + 1 Conference = 7
# 2024/25 +   : 5 CL + 1 EL + 1 Conference = 7
def get_european_spots(season: str) -> dict:
    """
    Returns {'cl': n, 'el': n, 'conf': n, 'total_european': n} for a given
    season string (e.g. "2023/2024", "2024/2025", "2024", or a single year).
    Handles both "YYYY/YYYY" and plain "YYYY" season formats.
    """
    season_str = str(season)
    # Extract the starting year of the season robustly
    if "/" in season_str:
        start_year = int(season_str.split("/")[0])
    else:
        start_year = int(season_str[:4])
 
    if start_year >= 2024:
        return {"cl": 5, "el": 1, "conf": 1, "total_european": 7}
    else:
        return {"cl": 4, "el": 2, "conf": 1, "total_european": 7}

And now lastly before proceeding, I will create a constant specifing the days within a match is considered to come in as a team pressure and also the day windows in which will define the team's congestion!

In [11]:
IMMINENT_MATCH_DAYS = 6

# For this two features there will be calculated the matches that a team has played in these day windows:
CONGESTION_WINDOW_7  = 7
CONGESTION_WINDOW_14 = 14

---

Enough with the constants, lets begin the actual work.

I am not going to explain everything in so much details because by now I have explained the whole process and the steps which the implementation will follow:

In [12]:
df = matches_interim_df.copy()

### Excluding the irrelevant competitions:

In [13]:
# Filter out competitions that are not part of our scope at all (qualifiers,
# national team competitions) — they have zero bearing on club-level pressure.
df = df[~df["competition_id"].isin(EXCLUDED_COMPETITION_IDS)]

### Applying the competitions priorities:

In [14]:
df = df.sort_values(["competition_id", "season", "date"]).reset_index(drop=True)
 
# Competition priority — only STANDARD and FINAL_ONLY ids are mapped;
# anything else (shouldn't exist after the exclusion filter) gets a low default.
df["competition_priority"] = (
    df["competition_id"].map(COMPETITION_PRIORITY).fillna(0.30)
)

### Creating an identifier for the standard and final competitions:

In [15]:
df["is_standard_competition"]  = df["competition_id"].isin(STANDARD_COMPETITION_IDS)
df["is_final_only_competition"] = df["competition_id"].isin(FINAL_ONLY_COMPETITION_IDS)

### Creating cumulative points and goal difference
I will calculate the points and the goal difference that each of the teams has, before the current match!Later this will help to identify how many points a team is behind the 1st, or how many points they need to get into top4, top6 or what is the point gap between the team and the relegation zone!

In [16]:
# Only the la liga matches:
league_matches = df[df["competition_id"] == LA_LIGA_ID].copy()

In [17]:
# Separating the matches into home and away ones - one row per team per match:

home_results = league_matches[[
    "game_id", "season", "date", "round",
    "home_club_id", "away_club_id", "home_club_goals", "away_club_goals",
]].rename(columns={
    "home_club_id": "team_id", "away_club_id": "opponent_id",
    "home_club_goals": "goals_for", "away_club_goals": "goals_against",
})
home_results["is_home"] = 1
 
away_results = league_matches[[
    "game_id", "season", "date", "round",
    "away_club_id", "home_club_id", "away_club_goals", "home_club_goals",
]].rename(columns={
    "away_club_id": "team_id", "home_club_id": "opponent_id",
    "away_club_goals": "goals_for", "home_club_goals": "goals_against",
})
away_results["is_home"] = 0
 
team_results = pd.concat([home_results, away_results], ignore_index=True)

In [18]:
# Defining the team's points and goal diff, based on the goals scored and conceded in the matches
# Win: 3 points
# Lose: 0 points
# Draw: 1 point
team_results["match_points"] = np.where(
    team_results["goals_for"] > team_results["goals_against"], 3,
    np.where(team_results["goals_for"] == team_results["goals_against"], 1, 0)
)

team_results["goal_diff"] = team_results["goals_for"] - team_results["goals_against"]

Calculating the cumulative points and goal diff before the matches:

In [19]:
team_results = team_results.sort_values(["team_id", "season", "date"]).reset_index(drop=True)

# Collecting the team's stats before their current match, using shift(1) -> no leakage:
team_results["cum_points"] = (
    team_results.groupby(["team_id", "season"])["match_points"]
    .transform(lambda s: s.shift(1).cumsum().fillna(0))
)
team_results["cum_gd"] = (
    team_results.groupby(["team_id", "season"])["goal_diff"]
    .transform(lambda s: s.shift(1).cumsum().fillna(0))
)
team_results["cum_goals_for"] = (
    team_results.groupby(["team_id", "season"])["goals_for"]
    .transform(lambda s: s.shift(1).cumsum().fillna(0))
)

team_results["matches_played"] = (
    team_results.groupby(["team_id", "season"]).cumcount()
)

### Creating the Team Standings table:

In [20]:
team_standings = team_results[[
    "game_id", "team_id", "season", "round",
    "cum_points", "cum_gd", "cum_goals_for", "matches_played",
]].copy()

### Defining the league state before each of the rounds:
I will create a function which defines the 1st team, top4 min points, top6/7 min points and relegation min points before each of the rounds using the state of the teams before entering the current round.When we define the team's points at the different positions in the league, we can then identify the state of each of the teams at the current round and eventually define how important is a certain match for a certain team!

In [21]:
# One row per (team, round) — each team's cumulative standing entering that round.
# A team may have multiple game_id entries for the same round only in edge cases
# (postponements); we take the standing as of their first match in that round,
# which represents the table state at the start of that round for everyone.
team_round_standing = (
    team_standings
    .sort_values(["season", "round", "team_id", "game_id"])
    .drop_duplicates(subset=["season", "round", "team_id"], keep="first")
)

In [22]:
team_standings.shape, team_round_standing.shape # As we can see there are no postponements or matches which have been played twice!

((8360, 8), (8360, 8))

In [23]:
def compute_table_reference_points(grp, season_value):
    """
        Given all teams' standings entering a round, compute reference points
        at the key cutoff positions (1st, CL cutoff, EL cutoff, relegation cutoff),
        using the season-appropriate European spot allocation.
    """
    
    spots = get_european_spots(season_value)
    cl_spot = spots["cl"] # last guaranteed CL position (Champions League)
    el_spot = spots["cl"] + spots["el"] # last guaranteed CL+EL position(League Europe)
    conf_spot = spots["cl"] + spots["el"] + spots["conf"] # last European position(COnference league)
 
    # Sorting the team's group so that we can get the cutoff positions easily:
    sorted_grp = grp.sort_values(
        ["cum_points", "cum_gd"], ascending=[False, False]
    ).reset_index(drop=True)
 
    n = len(sorted_grp)
 
    # This function returns the points of the teams, based on the provided position(e.g: pos = 1 -> 1st team points)
    def pts_at(pos):
        idx = min(pos - 1, n - 1)
        return sorted_grp.loc[idx, "cum_points"] if n >= 1 else np.nan
 
    return pd.Series({
        "table_pts_1st": pts_at(1),
        "table_pts_cl_cutoff": pts_at(cl_spot) if n >= cl_spot else np.nan,
        "table_pts_el_cutoff": pts_at(el_spot) if n >= el_spot else np.nan,
        "table_pts_conf_cutoff": pts_at(conf_spot) if n >= conf_spot else np.nan,
        "table_pts_rel_cutoff": pts_at(n - RELEGATION_ZONE_SIZE + 1) if n > RELEGATION_ZONE_SIZE else np.nan,
        "teams_in_table": n
    })

### Applying the function and creating the reference table

In [24]:
table_ref_rows = []
for (season_val, round_val), grp in team_round_standing.groupby(["season", "round"]):
    ref = compute_table_reference_points(grp, season_val)
    ref["season"] = season_val
    ref["round"]  = round_val
    table_ref_rows.append(ref)
 
table_refs = pd.DataFrame(table_ref_rows).reset_index(drop=True)

In [25]:
table_refs.columns

Index(['table_pts_1st', 'table_pts_cl_cutoff', 'table_pts_el_cutoff',
       'table_pts_conf_cutoff', 'table_pts_rel_cutoff', 'teams_in_table',
       'season', 'round'],
      dtype='str')

In [26]:
table_refs.duplicated(subset=['season', 'round']).any()

np.False_

In [27]:
# Attach season-specific spot counts to table_refs for downstream use:
spot_lookup = (
    table_refs[["season"]]
    .drop_duplicates()
    .assign(_spots=lambda d: d["season"].apply(get_european_spots))
)

spot_lookup["cl_spots"] = spot_lookup["_spots"].apply(lambda d: d["cl"])
spot_lookup["el_spots"] = spot_lookup["_spots"].apply(lambda d: d["el"])
spot_lookup["conf_spots"] = spot_lookup["_spots"].apply(lambda d: d["conf"])
spot_lookup = spot_lookup.drop(columns="_spots")
 
table_refs = table_refs.merge(spot_lookup, on="season", how="left")

### Pay attention how the European spots change in the features:

In [28]:
table_refs[['cl_spots', 'el_spots', 'conf_spots']]

,cl_spots,el_spots,conf_spots
0,4,2,1
1,4,2,1
2,4,2,1
3,4,2,1
4,4,2,1
...,...,...,...
413,5,1,1
414,5,1,1
415,5,1,1
416,5,1,1


### Creating the season progress factor
As I said into the beggining, the time is one of the most important factors that we must take into account when creating the features.What I want to achive with this function is to put exponential weighted importance on the matches as the season progress!I have already explained why this should be done!

However, for this functionallity I will use an algorithm which calculates a non-linear, smoothly scaling progress multiplier between **0 and 1** for a season based on the current round. Instead of increasing steadily by a fixed amount every round, the progress factor stays near **0** during the **early weeks**, accelerates quickly in the **middle** of the season, and **plateaus near** **1** as the season **concludes**!

In [29]:
def season_progress(round_num, total_rounds=SEASON_ROUNDS):
    """Sigmoid progress factor in [0, 1]. ~0 before round 10, ~1 by round 35."""
    
    # Convert to Linear Scale
    linear = round_num / total_rounds
    
    # Apply Sigmoid Transformation
    sig = 1 / (1 + np.exp(-10 * (linear - 0.55)))
    
    # Establishing the absolite boundaries
    sig_min = 1 / (1 + np.exp(-10 * (1/38  - 0.55)))
    sig_max = 1 / (1 + np.exp(-10 * (38/38 - 0.55)))
    
    # Returning the normalized range:
    return (sig - sig_min) / (sig_max - sig_min)

I have cretaed the function now because it will be needed for the creation of the upcoming features:

### Defining the team's fixture shedule:
Something which is very indiative for a team pressure is the amount of matches that a team has played before their current match.This identifies the team's congestion and also how likely a team is to have rotations in the squad(if the team want to keep his strength for a more important match) from which also depends the outcome of the match!

In [30]:
# I should again split the matches into home and away teams:

all_home = df[[
    "game_id", "date", "season", "home_club_id",
    "competition_id", "competition_priority",
    "is_standard_competition", "is_final_only_competition",
]].rename(columns={"home_club_id": "team_id"})
 
all_away = df[[
    "game_id", "date", "season", "away_club_id",
    "competition_id", "competition_priority",
    "is_standard_competition", "is_final_only_competition",
]].rename(columns={"away_club_id": "team_id"})
 
all_schedule = (
    pd.concat([all_home, all_away], ignore_index=True)
    .sort_values(["team_id", "date"])
    .reset_index(drop=True)
)

all_schedule["row_id"] = np.arange(len(all_schedule))

Now I will create the function which will accept the full teams matches shedule and for each of the teams it will identify how many matches has the team played in the different time windows(7, 14)!Of course the function will identify the amount of played matches before the team's current match!

In [31]:
# Congestion load (matches_last_7d / 14d) is computed from STANDARD
# competitions ONLY — a Supercopa or UEFA Super Cup final does not
# represent a "normal" extra fixture burden, it's a rare bonus/prestige
# fixture, so including it in congestion would distort the rotation signal.

# Self-merge each team's schedule against itself, keep only pairs within the
# window and strictly before the reference date, then count per reference row.
# This is the standard vectorised approach for "count prior events in window".
def vectorised_rolling_count(schedule: pd.DataFrame, window_days: int, count_col_name: str):
    """
        For every row in `schedule`, count how many other rows for the same
        team_id fall strictly within [date - window_days, date).
        Fully vectorised via self-merge on team_id + a date-range boolean mask.
    """
    left  = schedule[["row_id", "team_id", "date"]].rename(columns={"date": "ref_date"})
    right = schedule[["team_id", "date"]].rename(columns={"date": "evt_date"})
 
    merged = left.merge(right, on="team_id", how="left")
    merged["in_window"] = (
        (merged["evt_date"] <  merged["ref_date"])
        & (merged["evt_date"] >= merged["ref_date"] - pd.Timedelta(days=window_days))
    )
    
    counts = (
        merged.groupby("row_id")["in_window"].sum()
        .rename(count_col_name)
    )
    
    return schedule.merge(counts, on="row_id", how="left").fillna({count_col_name: 0})

Now I will apply the function using only the teams matches shedules from the **standard competitions**!

In [32]:
standard_schedule = all_schedule[all_schedule["is_standard_competition"]].copy()

In [33]:
standard_schedule = vectorised_rolling_count(standard_schedule, CONGESTION_WINDOW_7,  "matches_last_7d")
standard_schedule = vectorised_rolling_count(standard_schedule, CONGESTION_WINDOW_14, "matches_last_14d")

Now I will seperate the newly created features into a dedicated congestion df!

In [34]:
congestion_df = standard_schedule[[
    "game_id", "team_id", "matches_last_7d", "matches_last_14d"
]].copy()

congestion_df["matches_last_7d"]  = congestion_df["matches_last_7d"].astype(int)
congestion_df["matches_last_14d"] = congestion_df["matches_last_14d"].astype(int)

### Creating a next-match lookup features
The next thing that I am going to implement will be identifiers about the team's next match date and next match **priority**.By looking at the future matches of the teams, we can understand how important is the current match of the team.For example if the team's next match is from the Champions League, the team is going to keep his power and more importantly he will not risk it's best players for a match which is with smaller priority than the next match.

When we know the team's next matches and how important this next match is for the team, we can also define the fixture congestion score!

Something important: By looking at the team's future match we do not introduce data leakage beacuse the shedule of the matches is defined long before the team's current match, so these matches are already known.And I will use them only to gather the date and the predifined priority of the match, based on the competition.I will not use any stats and results from the future matches!

In [35]:
# Getting all of the sheduled matches from all of the competitions!
all_schedule_sorted = all_schedule.sort_values(["team_id", "date", "game_id"]).reset_index(drop=True)

In [36]:
# Next match of ANY kind (standard or final-only)
all_schedule_sorted["next_match_date"] = (
    all_schedule_sorted.groupby("team_id")["date"].shift(-1)
)
all_schedule_sorted["next_match_priority"] = (
    all_schedule_sorted.groupby("team_id")["competition_priority"].shift(-1)
)
all_schedule_sorted["next_match_game_id"] = (
    all_schedule_sorted.groupby("team_id")["game_id"].shift(-1)
)

# To ensure safety: if shift somehow produced the same game_id (duplicate date row),
# null it out rather than risk leakage
same_game_mask = all_schedule_sorted["next_match_game_id"] == all_schedule_sorted["game_id"]
all_schedule_sorted.loc[same_game_mask, ["next_match_date", "next_match_priority"]] = pd.NaT, np.nan

Now with the code above we got the next match of any kind.Now lets get the next non-league match:

In [37]:
# Getting the next non-league match by filtering to non-ES1 rows
# Then I will map each La Liga row to the nearest following non-league row via merge_asof
nonleague_schedule = (
    all_schedule_sorted[
        all_schedule_sorted["competition_id"] != LA_LIGA_ID
    ][["team_id", "date", "competition_priority"]]
    .rename(columns={"date": "nl_date", "competition_priority": "nl_priority"})
    .sort_values("nl_date")
    .reset_index(drop=True)
)

Now I will perform `merge asof` to assign the nearest non-league match to the La Liga matches.For the merge I will use `direction=forward` as also `allow_exact_matches=False`, so that the merge to happen only for future matches and also to not use matches at the same dates as the current team matches!

In [38]:
# merge_asof forward: for each (team_id, date) find the next non-league date >= date
# direction="forward" with allow_exact_matches=False ensures strictly-after lookup
left_for_asof = (
    all_schedule_sorted[["row_id", "team_id", "date"]]
    .sort_values("date")
    .reset_index(drop=True)
)

# The merge asof requires the datasets to be sorted only by the on key column
# Both of the datasets are sorted only by date, to ensure that the merge asof will be working correctly.
next_nonleague = pd.merge_asof(
    left_for_asof,
    nonleague_schedule.rename(columns={"date": "date"} if False else {}),
    left_on="date", 
    right_on="nl_date",
    by="team_id",
    direction="forward",
    allow_exact_matches=False,
)

In [39]:
next_nonleague = next_nonleague.rename(columns={
    "nl_date": "next_nonleague_date", "nl_priority": "next_nonleague_priority"
})

Now lets merge the next non-league matches with the full team's matches shedule:

In [40]:
all_schedule_sorted = all_schedule_sorted.merge(
    next_nonleague[["row_id", "next_nonleague_date", "next_nonleague_priority"]],
    on="row_id", how="left",
)

In [41]:
all_schedule_sorted.info()

<class 'pandas.DataFrame'>
RangeIndex: 19146 entries, 0 to 19145
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   game_id                    19146 non-null  int32         
 1   date                       19146 non-null  datetime64[us]
 2   season                     19146 non-null  category      
 3   team_id                    19146 non-null  int32         
 4   competition_id             19146 non-null  category      
 5   competition_priority       19146 non-null  float64       
 6   is_standard_competition    19146 non-null  bool          
 7   is_final_only_competition  19146 non-null  bool          
 8   row_id                     19146 non-null  int64         
 9   next_match_date            18599 non-null  datetime64[us]
 10  next_match_priority        18599 non-null  float64       
 11  next_match_game_id         18599 non-null  float64       
 12  next_nonleague_

### Combining the congestion features with the shedule matches and creating more features
Now I will merge the `all_shedule_sorted` with the `congestion_df` that we have created earlier and then I will create some more features such as: days_until_next_match, days_until_next_nonleague_match and at the end I will define the **team's fixture congestion score** using the amount of matches that the team has played in the last 7/14 days!

In [42]:
schedule_features = all_schedule_sorted.copy()

In [43]:
schedule_features = schedule_features.merge(
    congestion_df, on=["game_id", "team_id"], how="left"
)

schedule_features["matches_last_7d"]  = schedule_features["matches_last_7d"].fillna(0).astype(int)
schedule_features["matches_last_14d"] = schedule_features["matches_last_14d"].fillna(0).astype(int)

Creating `days_until_next_match` and `days_until_next_nonleague_match` features:

In [44]:
# I am going to clip the data in range 0-99, so that, if a team next match is after 100 or more days
# to not introduce very big day ranges and eventually confuse the models.

# For the teams that do not have next match I will fill them with 99 so that it is understandable that the team
# next match is not close and their will be no impact of the team's future matches for the team's rotation risk for example!
schedule_features["days_until_next_match"] = (
    (schedule_features["next_match_date"] - schedule_features["date"])
    .dt.days.fillna(99).clip(0, 99)
)

schedule_features["days_until_next_nonleague"] = (
    (schedule_features["next_nonleague_date"] - schedule_features["date"])
    .dt.days.fillna(99).clip(0, 99)
)

As the clip will assign zeros for the matches that are with negative difference(invalid records), I want to ensure that there are no records with zero dates diff, because this would mean that there is a mistake, either in the data or in our calculations: 

In [45]:
schedule_features[
    (schedule_features.days_until_next_match == 0) |
    (schedule_features.days_until_next_nonleague == 0)
].shape

(0, 18)

#### Defining important_match_ahead feature
I will create the `important_match_ahead` feature so that to identify if the team has an important match in the near future!

In [46]:
schedule_features["next_match_priority"] = schedule_features["next_match_priority"].fillna(0.0)
schedule_features["next_nonleague_priority"] = schedule_features["next_nonleague_priority"].fillna(0.0)

# An match will be considered important, if he is within the range of 6 days after the current match
# and also the match is not from LaLiga and it's priority is more then 0.55 which includes:
# Champions League, League Europe, Conference League, Copa Del Ray and the EUFA Super Cup:
schedule_features["important_match_ahead"] = (
    (schedule_features["days_until_next_nonleague"] <= IMMINENT_MATCH_DAYS)
    & (schedule_features["next_nonleague_priority"] >= 0.55)
).astype(int)

In [47]:
schedule_features[schedule_features.important_match_ahead == 1].shape

(2045, 19)

### Defining the team's fuxture congestion score and the identifier for rotation risk:
About the team's rotation risk: \
By rotation risk I mean the chance of a team to make changes in his squad and in it's intentions at all!A rotation risk will be identified using the team's fixture_congestion_score(team is overloaded) a also the team next nonleague match and the days until it comes.This way, based in the matches that the team has played recently and the important matches that he has ahead, we will identify the chance of the team to make some changes and rotate his squad maybe - by rotate I mean keeping it's key players and play with others not so important players, in order to keep the team power for the more important matches!

In [48]:
# Team fixture congestion score is created using the team's matches in the last 7/14 days to identify 
# how tired and overloaded the team might be: 
schedule_features["fixture_congestion_score"] = (
    schedule_features["matches_last_7d"] * 0.60 # Assing more weight on the more recent matches
    + schedule_features["matches_last_14d"] * 0.40
).clip(0, 10) / 10.0
 
# The rotation risk predicts the current risk of players being benched. 
# It blends past fatigue (50% weight) with the pressure of the next upcoming non-league match (50% weight).

# Exponential Decay (np.exp): 
# The threat of an upcoming cup match grows exponentially the closer it gets.
# If a massive game (next_nonleague_prio is high) is tomorrow, the math evaluates close to 1.0, 
# keeping the priority score high.

# If that game is 3 weeks away, the math decays the priority score close to 0.0, 
# because managers do not rest players for games far in the future.
schedule_features["rotation_risk"] = (
    schedule_features["fixture_congestion_score"] * 0.50
    + (
        schedule_features["next_nonleague_priority"]
       * np.exp(-schedule_features["days_until_next_nonleague"] / 7.0)
    ) * 0.50
).clip(0.0, 1.0)

### Dealing with the final competitions.
As we said in the beggining, we separate the competitions into standard and final competitions.The final competitions are more specific as they are played between 2-4 teams and are executed into 1 to 2 rounds only!

What I will do is that I will create some identifiers if a team is about to participate in such competition?For each team, we will simply ask the question: **Was this team in a *Supercopa* / *UEFA Super Cup* final in the last N days?**

In [49]:
# Defining the maximum days range that the final cup match will be considered 
# important and will have an impact over for the team current match!
# When creating the feature, this value will be subtracted from the team current match date
# and if the final cup match has been played before the subtracted dates result, the final cup match 
# will be considered: 'in_window' 
CUP_FINAL_LOOKBACK_DAYS = 10

In [50]:
# Getting only the matches from the final competitions:
final_only_schedule = all_schedule[all_schedule["is_final_only_competition"]].copy()
final_only_schedule = final_only_schedule[["team_id", "date"]].rename(
    columns={"date": "final_date"}
)

In [51]:
# Merging with the full matches schedule:
cup_final_check = all_schedule[["row_id", "team_id", "date"]].merge(
    final_only_schedule, on="team_id", how="left"
)

In [52]:
# Defining if the final cup match is within the days window, to be considered important for the team:
cup_final_check["in_window"] = (
    (cup_final_check["final_date"] <  cup_final_check["date"])
    & (cup_final_check["final_date"] >= cup_final_check["date"] - pd.Timedelta(days=CUP_FINAL_LOOKBACK_DAYS))
)

In [53]:
# Creating the feature which will identify if the final cup match has been recent to the team current match: 
cup_final_recent = (
    cup_final_check.groupby("row_id")["in_window"].any()
    .rename("cup_final_recent").astype(int)
)

#### Merging the `cup_final_recent` with the `schedule_features`
Adding the newly created `cup_final_recent` features to the other team matches features:

In [54]:
schedule_features = schedule_features.merge(cup_final_recent, on="row_id", how="left")
schedule_features["cup_final_recent"] = schedule_features["cup_final_recent"].fillna(0).astype(int)

In [55]:
schedule_features[schedule_features.cup_final_recent == 1].shape

(94, 22)

The final result of the `schedule_features`:

In [56]:
schedule_final = schedule_features[[
    "game_id", "team_id",
    "matches_last_7d", "matches_last_14d",
    "days_until_next_match", "days_until_next_nonleague",
    "next_match_priority", "next_nonleague_priority",
    "important_match_ahead", "fixture_congestion_score", "rotation_risk",
    "cup_final_recent",
]].copy()

---

### Building match level importance features for the La Liga matches:
Now I will proceed by separating the La Liga matches and creating more features about the matches importance:

In [57]:
# Getting the La Liga matches only:
la_liga = league_matches[[
    "game_id", "season", "round", "date",
    "home_club_id", "away_club_id",
    "home_club_position", "away_club_position",
]].copy()

# Merging the La Liga matches with the team standings data(cumulative goals, points, minutes)
# The merge is made on the team_id and game_id so that to get the standing of every team in a La Liga match
# from the team_standings dataset.
# The data is split into home and away matches. 
la_liga = la_liga.merge(
    team_standings.rename(columns={
        "team_id": "home_club_id", "cum_points": "home_cum_points",
        "cum_gd": "home_cum_gd", "cum_goals_for": "home_cum_goals_for",
        "matches_played": "home_matches_played",
    })[["game_id", "home_club_id", "home_cum_points", "home_cum_gd",
        "home_cum_goals_for", "home_matches_played"]],
    on=["game_id", "home_club_id"], how="left",
).merge(
    team_standings.rename(columns={
        "team_id": "away_club_id", "cum_points": "away_cum_points",
        "cum_gd": "away_cum_gd", "cum_goals_for": "away_cum_goals_for",
        "matches_played": "away_matches_played",
    })[["game_id", "away_club_id", "away_cum_points", "away_cum_gd",
        "away_cum_goals_for", "away_matches_played"]],
    on=["game_id", "away_club_id"], how="left",
)
 
# Merge table reference points keyed by (season, round)
# The table refs table contains the different team positions in the league before each of the rounds!
# With these merge we get the points of the 1st, min points for top 4/7, relegation points etc!
la_liga = la_liga.merge(table_refs, on=["season", "round"], how="left")

### Adding season progress factor and identifing if the match is at the meaningful season phase
If you remember we have created the `season_progress` function which based on the current match round and all of the rounds and applies a season progress factor, using a sigmoid algorithm.For all of the rounds before the 10th the function wil simply return progress factor near 0.However, as the season progress and the round exceed the 10th round the function quickly starts to return higher and higher values as season progress!

The othet thing which we will identify is: Is the match round meaningful.This will be identified based in the `PRESSURE_ACTIVATION_ROUND` constant that we have defined in the beggining, which states that afte the round exceed 10, the pressure should be activated and therefore, the round should be considered meaningful!

These two things are required so that a match which is played in the very beggining of the season is not so important as a match which is played in the last rounds - where every team is under pressure and is fighting for something - This is my idea!

Now, before appling this function, I will need to transform the round features, because currently the rounds are represented in format: **n. Matchday**, where **n** is the round!So lets transform the feature, by leaving only the round number.This should be done because the `season_progress` function require integer as a round to work!

In [58]:
la_liga['round'] = la_liga['round'].apply(lambda x: transform_match_round(x))

In [59]:
la_liga['round'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38])

In [60]:
la_liga["season_progress"] = la_liga["round"].apply(season_progress)
la_liga["season_meaningful"] = (la_liga["round"] >= PRESSURE_ACTIVATION_ROUND).astype(int)

Now with a for loop I will iterate over each of the teams in the La Liga matches and based on the points of the teams I will calculate how many points each of the teams is away from the different positions in the league.

You see, here the table_refs table that we have created with the pre rounds teams positions will get in use!

In [61]:
# The for loop iterates over each of the teams(home/away) and uses the team's current league position 
# and points gathered at the moment of match to calculate how far away are the team to the different 
# important position in the league(title, el cutoff, cl cutoff, relegation) 
for side, pos_col, pts_col in [
    ("home", "home_club_position", "home_cum_points"),
    ("away", "away_club_position", "away_cum_points"),
]:
    p = la_liga[pts_col]
    la_liga[f"{side}_pts_behind_1st"] = (la_liga["table_pts_1st"] - p).clip(lower=0)
    la_liga[f"{side}_pts_to_cl"] = la_liga["table_pts_cl_cutoff"] - p
    la_liga[f"{side}_pts_to_el"] = la_liga["table_pts_el_cutoff"] - p
    la_liga[f"{side}_pts_to_conf"] = la_liga["table_pts_conf_cutoff"] - p
    la_liga[f"{side}_pts_above_relegation"] = p - la_liga["table_pts_rel_cutoff"]
    la_liga[f"{side}_league_position"] = la_liga[pos_col]

### Creating pressure flags and scores:
Now for each of the teams I will calculate different pressure flags using the features that we have created above.For example a team will be considered to fight for the title if the gap between him and the 1st is less than or equal 5.I will create flags for all of the different important league positions that a team might want to fight for: title flag, cl flag, el flag, relegation flag!

It is also very important to say that the pressure flags will account for the season progress factor.And this is because we cannot consider a team, fighting for any league position before the season to have been progressed in a sufficient level!

In [62]:
for side in ["home", "away"]:
    prog = la_liga["season_progress"]
    meaningful = la_liga["season_meaningful"]
 
    # Creating the team position flags - are they fighting for something:
    # Title:
    la_liga[f"{side}_title_race_flag"] = (
        meaningful
        & (la_liga[f"{side}_pts_behind_1st"] <= TITLE_RACE_GAP_MAX)
        & (la_liga[f"{side}_pts_behind_1st"] >= 0)
        & (la_liga[f"{side}_league_position"] <= 4)
    ).astype(int)
 
    # Champions League:
    la_liga[f"{side}_cl_race_flag"] = (
        meaningful
        & (la_liga[f"{side}_pts_to_cl"].between(-2, TOP4_GAP_MAX))
        & (la_liga[f"{side}_league_position"].between(2, 7))
    ).astype(int)
 
    # League Europe:
    la_liga[f"{side}_el_race_flag"] = (
        meaningful
        & (la_liga[f"{side}_pts_to_el"].between(-2, TOP6_GAP_MAX))
        & (la_liga[f"{side}_league_position"].between(3, 9))
        & (la_liga[f"{side}_cl_race_flag"] == 0) # A team cannot have both cl_race_flag and el_race_flag - This is very important to be specified 
    ).astype(int)
 
    # Relegation Zone:
    la_liga[f"{side}_relegation_flag"] = (
        meaningful
        & (la_liga[f"{side}_pts_above_relegation"] <= RELEGATION_GAP_MAX)
    ).astype(int)
 
    # Creating the pressure scores - season_progress is applied only once
    la_liga[f"{side}_title_pressure"] = (
        la_liga[f"{side}_title_race_flag"]
        * (1 - la_liga[f"{side}_pts_behind_1st"] / (TITLE_RACE_GAP_MAX + 1))
        * prog
    ).clip(0, 1)
 
    la_liga[f"{side}_cl_pressure"] = (
        la_liga[f"{side}_cl_race_flag"]
        * (1 - la_liga[f"{side}_pts_to_cl"].clip(0) / (TOP4_GAP_MAX + 1))
        * prog
    ).clip(0, 1)
 
    la_liga[f"{side}_el_pressure"] = (
        la_liga[f"{side}_el_race_flag"]
        * (1 - la_liga[f"{side}_pts_to_el"].clip(0) / (TOP6_GAP_MAX + 1))
        * prog
    ).clip(0, 1)
 
    la_liga[f"{side}_relegation_pressure"] = (
        la_liga[f"{side}_relegation_flag"]
        * (1 - la_liga[f"{side}_pts_above_relegation"].clip(0) / (RELEGATION_GAP_MAX + 1))
        * prog
    ).clip(0, 1)

Now as we have created the team's flags and idenified for what each team is fighting for at the time of the matches, now lest also identify the direct confrontation matches, where teams which are fighting for the same thing meet together:

### Creating direct confrontation features:

In [63]:
la_liga["title_direct_clash"] = (
    la_liga["home_title_race_flag"] & la_liga["away_title_race_flag"]
).astype(int)

la_liga["cl_direct_clash"] = (
    la_liga["home_cl_race_flag"] & la_liga["away_cl_race_flag"]
).astype(int)

la_liga["el_direct_clash"] = (
    la_liga["home_el_race_flag"] & la_liga["away_el_race_flag"]
).astype(int)

la_liga["relegation_direct_clash"] = (
    la_liga["home_relegation_flag"] & la_liga["away_relegation_flag"]
).astype(int)

# These refers to a match where a relegation-free team meets a possibly relegated team
# This feature can be very useful identifire because in this situation, the team which is threatened
# to be relegated will try to give his best to win the meatch and eventually will have more ambition and
# power to win it than the other relagation-free team!
la_liga["relegation_six_pointer"] = (
    (la_liga["home_relegation_flag"] | la_liga["away_relegation_flag"])
    & ~(la_liga["home_relegation_flag"] & la_liga["away_relegation_flag"])
).astype(int)

### Creating direct position clash identifier

In [64]:
# Position gap between the teams at the time of the match
la_liga["position_gap"] = (la_liga["home_club_position"] - la_liga["away_club_position"]).abs()

# Points gap between the teams at the time of the match
la_liga["points_gap_between_teams"] = (
    la_liga["home_cum_points"] - la_liga["away_cum_points"]
).abs()

In [65]:
la_liga["direct_position_clash"] = (
    la_liga["season_meaningful"]
    & (la_liga["position_gap"] <= 2)
    & (
        la_liga["title_direct_clash"]
        | la_liga["cl_direct_clash"]
        | la_liga["el_direct_clash"]
        | la_liga["relegation_direct_clash"]
        | ((la_liga["home_club_position"] <= 7) & (la_liga["away_club_position"] <= 7)) # Conference league direct clash
    )
).astype(int)

### Merging schedule / rotation / cup-final features for both home and away team sides:

In [66]:
la_liga = la_liga.merge(
    schedule_final.rename(columns={
        col: f"home_{col}" for col in schedule_final.columns
        if col not in ["game_id", "team_id"]
    }).rename(columns={"team_id": "home_club_id"}),
    on=["game_id", "home_club_id"], how="left",
).merge(
    schedule_final.rename(columns={
        col: f"away_{col}" for col in schedule_final.columns
        if col not in ["game_id", "team_id"]
    }).rename(columns={"team_id": "away_club_id"}),
    on=["game_id", "away_club_id"], how="left",
)

### Creating the final match-importance feature using all of the features that have been cretaed by now:
Now as we have everything we need, it is time to create the final feature which will define the importance of the teams matches!

The match-importance feature will be a composite score which combines all of the features that we have created, including the pressure scores, flags, points, fixture congestion, rotation risk.

Here are the weights that will be applied for each of the features:
- 0.40 — highest pressure race the team is in (already season-weighted)
- 0.30 — direct confrontation bonus (already season-weighted via clash_bonus)
- 0.20 — table closeness (already season-weighted via table_tightness)
- -0.10 — rotation penalty (reduces importance if squad likely to be rotated)

ALl of these components use the season progress factor!

In [67]:
for side in ["home", "away"]:
    
    # We collect all of the pressure of the team:
    race_pressure = (
        la_liga[f"{side}_title_pressure"].fillna(0) * 1.00
        + la_liga[f"{side}_cl_pressure"].fillna(0) * 0.80
        + la_liga[f"{side}_el_pressure"].fillna(0) * 0.60
        + la_liga[f"{side}_relegation_pressure"].fillna(0)* 0.90
    ).clip(0, 1)
 
    # clash_bonus carries its own single season_progress multiplication
    clash_bonus = (
        la_liga["title_direct_clash"] * 1.0
        + la_liga["cl_direct_clash"] * 0.8
        + la_liga["el_direct_clash"] * 0.8
        + la_liga["relegation_direct_clash"] * 0.9
        + la_liga["direct_position_clash"] * 0.7
    ).clip(0, 1) * la_liga["season_progress"]
 
    rotation_penalty = la_liga[f"{side}_rotation_risk"].fillna(0) * 0.30
 
    # How tight is the points distribution in the league
    # If there is very little gap in the points between the 1st and Europe League cutoff(entrance of European places)
    # it means that the teams are equally strong and there are not defined favorites - This makes the matches much
    # more important and highly-pressured!
    # table_tightness carries its own single season_progress multiplication
    table_tightness = (
        1 - (
            (la_liga["table_pts_1st"] - la_liga["table_pts_el_cutoff"])
            .clip(0, 18) / 18
        )
    ).fillna(0.5) * la_liga["season_progress"]
 
    # Creating the final composite feature:
    # The rotation penalty will decrease the importance of the match, 
    # because if a team is likely to rotate it's squad, it means that the match
    # is not so important and a more important one is coming
    la_liga[f"{side}_match_importance"] = (
        race_pressure * 0.40
        + clash_bonus * 0.30
        + table_tightness * 0.20
        - rotation_penalty * 0.10
    ).clip(0.0, 1.0)

### Final features list:

In [68]:
PER_TEAM_FEATURES = [
    "cum_points", "cum_gd", "matches_played",
    "pts_behind_1st", "pts_to_cl", "pts_to_el", "pts_to_conf", "pts_above_relegation",
    "league_position", "title_race_flag", "cl_race_flag", "el_race_flag", "relegation_flag",
    "title_pressure", "cl_pressure", "el_pressure", "relegation_pressure",
    "matches_last_7d", "matches_last_14d", "days_until_next_match", "days_until_next_nonleague",
    "next_match_priority", "next_nonleague_priority", "important_match_ahead", "fixture_congestion_score", 
    "rotation_risk", "cup_final_recent", "match_importance"
]
 
MATCH_LEVEL_FEATURES = [
    "season_progress", "season_meaningful",
    "title_direct_clash", "cl_direct_clash", "el_direct_clash", "relegation_direct_clash",
    "relegation_six_pointer", "direct_position_clash",
    "position_gap", "points_gap_between_teams",
    "cl_spots", "el_spots", "conf_spots",
]

In [69]:
all_cols = (
    ["game_id"]
    + [f"home_{f}" for f in PER_TEAM_FEATURES]
    + [f"away_{f}" for f in PER_TEAM_FEATURES]
    + MATCH_LEVEL_FEATURES
)
 
output = la_liga[all_cols].copy()

### Checking for missing values:

In [70]:
output[output.isna()].any().any()

np.False_

In [71]:
output.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 70 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   game_id                         4180 non-null   int32  
 1   home_cum_points                 4180 non-null   float64
 2   home_cum_gd                     4180 non-null   float64
 3   home_matches_played             4180 non-null   int64  
 4   home_pts_behind_1st             4180 non-null   float64
 5   home_pts_to_cl                  4180 non-null   float64
 6   home_pts_to_el                  4180 non-null   float64
 7   home_pts_to_conf                4180 non-null   float64
 8   home_pts_above_relegation       4180 non-null   float64
 9   home_league_position            4180 non-null   Int8   
 10  home_title_race_flag            4180 non-null   int64  
 11  home_cl_race_flag               4180 non-null   int64  
 12  home_el_race_flag               4180 non-null

### Merging the `output` with the `matches_interim_df`:

In [72]:
matches_interim_df = matches_interim_df.merge(output, on="game_id", how="left")

### Getting only the La Liga matches
It is time to say goodbay to the other competitions and leave only the matches from La Liga.As you should already know, out target are the La Liga matches and nothing else.All of the other data was used for the creation of the features and the data has been extremely useful and important.However, we don't need the data any more and we should get only the target matches:

In [73]:
target_matches = output.game_id.unique()

In [74]:
matches_interim_df = matches_interim_df[matches_interim_df.game_id.isin(target_matches)]

In [75]:
matches_interim_df.shape

(4180, 128)

### Validating the team matches:

In [76]:
matches_interim_df.groupby('season').game_id.count()

season
2014/2015    380
2015/2016    380
2016/2017    380
2017/2018    380
2018/2019    380
2019/2020    380
2020/2021    380
2021/2022    380
2022/2023    380
2023/2024    380
2024/2025    380
Name: game_id, dtype: int64

In [77]:
validate_team_matches(
    data=matches_interim_df, 
    home_team_title='home_club_name', 
    away_team_title='away_club_name'
)

{'2014/2015': 20,
 '2015/2016': 20,
 '2016/2017': 20,
 '2017/2018': 20,
 '2018/2019': 20,
 '2019/2020': 20,
 '2020/2021': 20,
 '2021/2022': 20,
 '2022/2023': 20,
 '2023/2024': 20,
 '2024/2025': 20}

Well everything seems to be perfect!

---
Now as we have created all of the features, it is time to make some little cleaning.
### Cleaning the data:

The most important thing that I should do is to change the teams names in the dataset and make them consistent with the other datasets!

In order to do so, I need the Understat matches dataset, which is the dataset that the project uses as a **standard**!

In [78]:
understat_matches = pd.read_parquet('../../data/interim/understat_data/interim_matches_v1.parquet')

Before proceeding I will also need to rename some of the columns so that to keep the consistency across the datasets.

The columns that I will rename are the Transfermarkt's **home_club_name and away_club_name**.I will rename them to **h_title, a_title**, just they are named in the Understat data!

### Renaming the team's names columns:

In [79]:
matches_interim_df = matches_interim_df.rename(columns={
    'home_club_name': 'h_title',
    'away_club_name': 'a_title'
})

Now the team's names columns are consistent with the Understat dataset columns!So, lets proceed with the team's names mapping:

Getting the unique teams names across all of the seasons from the datasets(Understat and Transfermarkt): 

In [80]:
understat_all_teams = (
    understat_matches
        .groupby('season')['h_title']
            .unique()
            .to_dict()
)

transfermarkt_data_all_teams = (
    matches_interim_df
        .groupby('season')['h_title']
            .unique()
            .to_dict()
)

In [81]:
transfermarkt_data_all_teams

{'2014/2015': ['Sevilla Fútbol Club S.A.D.', 'Levante Unión Deportiva S.A.D.', 'Rayo Vallecano de Madrid S. A. D.', 'Real Club Celta de Vigo S. A. D.', 'Futbol Club Barcelona', ..., 'Club Atlético de Madrid S.A.D.', 'Deportivo de La Coruña', 'Elche Club de Fútbol S.A.D.', 'Villarreal Club de Fútbol S.A.D.', 'Athletic Club Bilbao']
 Length: 20
 Categories (921, str): ['1. FC Slovácko', '1. Fußball- und Sportverein Mainz 05', '1. Fußball-Club Köln', '1. Fußballclub Heidenheim 1846', ..., 'Östersunds FK', 'Újpest FC', 'İstanbul Başakşehir Futbol Kulübü', 'Футбольный клуб "Локомотив" Москва'],
 '2015/2016': ['Athletic Club Bilbao', 'Málaga CF', 'Rayo Vallecano de Madrid S. A. D.', 'Levante Unión Deportiva S.A.D.', 'Reial Club Deportiu Espanyol de Barcelona S.A..., ..., 'Getafe Club de Fútbol S. A. D. Team Dubai', 'Villarreal Club de Fútbol S.A.D.', 'Real Madrid Club de Fútbol', 'Real Sociedad de Fútbol S.A.D.', 'Valencia Club de Fútbol S. A. D.']
 Length: 20
 Categories (921, str): ['1. FC

As we can see, there are 20 unique teams in each of the seasons!

Now as we have collected all of the teams names for each of the seasons for the two datasets, now lets proceed by using the `build_team_mapping` function which will automatically create a **dictionary** with the invalid names of the **Transfermarkt data** teams and the corresponding valid **Understat teams** names!

For the function implemantation: [teams_names_mapper](../../src/football_betting_analysis/data/teams_names_mapper.py)

Before using the algorithm we should first create a **normalization function** which will remove any spaces and numeric symbols and replace them with single spaces!For this specific dataset some names are with accents such as: Almer**í**a.In order for the matching to work, the normalizations function should remove these accents and leave only valid unicode symbols!Also the function will remove some collocations such as: **Futbol Club, Club de Fútbol, de Fútbol** - These are commonly used in the teams names in the Transfermarkt dataset and if we leave them, they will make problems to the team names mapping algorithm!

In [82]:
def normalize(text: str) -> str:
    text = text.lower()
    
    #2.Removing words like S.A.D. (e.g. "s.a.d.", "s. a. d.")
    text = re.sub(r'\bs\.?\s*a\.?\s*d\.?\b', '', text)
    
    #3.Removing "futbol" (including the accent 'ú')
    text = re.sub(r'\bclub\s+de\s+f[uú]tbol\b', '', text)
    text = re.sub(r'\bfutbol\s+club\b', '', text)
    text = re.sub(r'\bde\s+f[uú]tbol\b', '', text)
    
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    nfkd_form = unicodedata.normalize('NFKD', text)
    text = ''.join([c for c in nfkd_form if not unicodedata.category(c) == 'Mn'])
    
    return text


In [83]:
# An example of how the normalize function works:
normalize('Real Sociedad de Fútbol S.A.D.')

'real sociedad'

### Building the team mapping dictionary:
Now lets use the `build_team_mapping` function to see how the algorithm will map the team names from the two datasets:

In [84]:
mapping, unresolved = build_team_mapping(
    normalize=normalize, 
    source_names=transfermarkt_data_all_teams['2014/2015'], 
    target_names=understat_all_teams['2014/2015'])

mapping

{'Sevilla Fútbol Club S.A.D.': 'Sevilla',
 'Levante Unión Deportiva S.A.D.': 'Levante',
 'Rayo Vallecano de Madrid S. A. D.': 'Rayo Vallecano',
 'Real Club Celta de Vigo S. A. D.': 'Celta Vigo',
 'Futbol Club Barcelona': 'Barcelona',
 'UD Almería': 'Almeria',
 'SD Eibar': 'Eibar',
 'Málaga CF': 'Malaga',
 'Granada CF': 'Granada',
 'Real Madrid Club de Fútbol': 'Real Madrid',
 'Valencia Club de Fútbol S. A. D.': 'Valencia',
 'Reial Club Deportiu Espanyol de Barcelona S.A.D.': 'Barcelona',
 'Getafe Club de Fútbol S. A. D. Team Dubai': 'Getafe',
 'Córdoba CF': 'Cordoba',
 'Real Sociedad de Fútbol S.A.D.': 'Real Sociedad',
 'Club Atlético de Madrid S.A.D.': 'Atletico Madrid',
 'Deportivo de La Coruña': 'Deportivo La Coruna',
 'Elche Club de Fútbol S.A.D.': 'Elche',
 'Villarreal Club de Fútbol S.A.D.': 'Villarreal',
 'Athletic Club Bilbao': 'Athletic Club'}

As we can see, there is a map for each of the teams, which is perfect.The only thing which should be fixed is that the team name: **'Barcelona'** is mapped more than ones!And this is happening because of this team names: **Reial Club Deportiu Espanyol de Barcelona S.A.D.** - As it seems the algorithm have failed to capture the **Espanyol** name and instead it has mapped this name to **Barcelona**, because it is also contained in the name!

In order to fix this, I will use the specific parameter: `aliases`, which is used for more specific team's names that the algorithm cannot capture so easily!So lets use the parameter:

In [85]:
# The algorithm works with the already normalized team's names, 
# so I need to get the normalized version of the name:
specific_team_name = normalize("Reial Club Deportiu Espanyol de Barcelona S.A.D.")

EXACT_ALIASES = {
    specific_team_name: 'Espanyol' # The right team name that we expect!
}

In [86]:
mapping, unresolved = build_team_mapping(
    normalize=normalize, 
    source_names=transfermarkt_data_all_teams['2014/2015'], 
    target_names=understat_all_teams['2014/2015'],
    aliases=EXACT_ALIASES)

mapping

{'Sevilla Fútbol Club S.A.D.': 'Sevilla',
 'Levante Unión Deportiva S.A.D.': 'Levante',
 'Rayo Vallecano de Madrid S. A. D.': 'Rayo Vallecano',
 'Real Club Celta de Vigo S. A. D.': 'Celta Vigo',
 'Futbol Club Barcelona': 'Barcelona',
 'UD Almería': 'Almeria',
 'SD Eibar': 'Eibar',
 'Málaga CF': 'Malaga',
 'Granada CF': 'Granada',
 'Real Madrid Club de Fútbol': 'Real Madrid',
 'Valencia Club de Fútbol S. A. D.': 'Valencia',
 'Reial Club Deportiu Espanyol de Barcelona S.A.D.': 'Espanyol',
 'Getafe Club de Fútbol S. A. D. Team Dubai': 'Getafe',
 'Córdoba CF': 'Cordoba',
 'Real Sociedad de Fútbol S.A.D.': 'Real Sociedad',
 'Club Atlético de Madrid S.A.D.': 'Atletico Madrid',
 'Deportivo de La Coruña': 'Deportivo La Coruna',
 'Elche Club de Fútbol S.A.D.': 'Elche',
 'Villarreal Club de Fútbol S.A.D.': 'Villarreal',
 'Athletic Club Bilbao': 'Athletic Club'}

Now everything seems to be perfect!

Making some validations to ensure that everything is correct:

In [87]:
counter = 0
for key, value in mapping.items():
    if value in understat_all_teams['2014/2015']:
        counter += 1
        
counter

20

In [88]:
seasons = [f"{str(year)}/{str(year+1)}" for year in range(START_SEASON, END_SEASON)]

valid_teams = {}
for season in seasons:
    season = str(season)
    source_names = transfermarkt_data_all_teams[season]
    target_names = understat_all_teams[season]
    
    # These are the results from the mapping algorithm
    mapping, unresolved = build_team_mapping(
        normalize=normalize, 
        source_names=source_names, 
        target_names=target_names,
        aliases=EXACT_ALIASES)
    
    counter = 0
    for key, value in mapping.items():
        if value in understat_all_teams[season]:
            counter += 1
            
    valid_teams[season] = counter

In [89]:
valid_teams

{'2014/2015': 20,
 '2015/2016': 20,
 '2016/2017': 20,
 '2017/2018': 20,
 '2018/2019': 20,
 '2019/2020': 20,
 '2020/2021': 20,
 '2021/2022': 20,
 '2022/2023': 20,
 '2023/2024': 20,
 '2024/2025': 20}

### Applying the Mapping function:

In [90]:
df = matches_interim_df.copy()

In [91]:
seasons = [f"{str(year)}/{str(year+1)}" for year in range(START_SEASON, END_SEASON)]

season_mappings = {}
season_unresolved = {}

for season in seasons:
    source_names = transfermarkt_data_all_teams[season]
    target_names = understat_all_teams[season]

    mapping, unresolved = build_team_mapping(
        normalize=normalize,
        source_names=source_names,
        target_names=target_names,
        aliases=EXACT_ALIASES
    )

    season_mappings[season] = mapping
    season_unresolved[season] = unresolved

Now we will use the mapping dicts that we have created from above and map the names row by row into the Transfermarkt dataset.

I will create a function which accepts a `team_type_column` which is the column of the team name in the match: either the **HomeTeam** or the **AwayTeam**.The function should apply the mapping for the both dataset columns!

In [92]:
def map_team(row, team_type_column: str):
    season = row['season']
    team = row[team_type_column]

    mapping = season_mappings.get(season, {})

    return mapping.get(team, None)

Now lets finally apply the mapping into the dataset, using the function above.

First I will apply the function for the Home teams and then for the Away ones:

In [93]:
df['h_title_clean'] = df.apply(lambda row: map_team(row, 'h_title'), axis=1)

In [94]:
df['a_title_clean'] = df.apply(lambda row: map_team(row, 'a_title'), axis=1)

In [95]:
df[['h_title_clean', 'a_title_clean']]

,h_title_clean,a_title_clean
1,Sevilla,Valencia
2,Levante,Villarreal
3,Rayo Vallecano,Atletico Madrid
4,Celta Vigo,Getafe
5,Barcelona,Elche
...,...,...
12933,Valencia,Real Valladolid
12934,Alaves,Villarreal
12935,Espanyol,Girona
12936,Getafe,Atletico Madrid


Now lets validated that everything has been ok:

In [96]:
df[
    (df['h_title_clean'].notna()) &
    (df['a_title_clean'].notna())
].shape

(4180, 130)

Now lets remove the helping columns and fill in the valid teams names into the original columns:

In [97]:
df['h_title'] = df['h_title_clean']
df['a_title'] = df['a_title_clean']

df = df.drop(columns='h_title_clean')
df = df.drop(columns='a_title_clean')

In [98]:
df.groupby('season')['h_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: h_title, dtype: int64

In [99]:
df.groupby('season')['a_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: a_title, dtype: int64

And now the most important validation: Are teams names in two datasets perfectly consistent?

In [100]:
valid_seasons = 0
for season in seasons:
    current_elo_matching_teams_names = sorted(df[df['season'] == season]['h_title'].unique())
    current_understat_teams_names = sorted(understat_matches[understat_matches['season'] == season]['h_title'].unique())
    
    if current_elo_matching_teams_names == current_understat_teams_names:
        valid_seasons += 1

valid_seasons

11

Ok everything is just perfect.

In [101]:
matches_interim_df = df

In [102]:
matches_interim_df.shape

(4180, 128)

Now the next thing I will do is to optimize the dataset memory usage:

### Optimizing the dataset memory usage:

In [103]:
matches_interim_df = optimize_dataframe_memory(matches_interim_df)

Initial Memory Usage: 3.83 MB
Final Memory Usage: 1.98 MB
Memory Reduction: 48.15%

Optimized Data Types:
game_id                              int32
competition_id                    category
season                            category
round                             category
date                        datetime64[us]
                                 ...      
position_gap                          int8
points_gap_between_teams           float32
cl_spots                           float32
el_spots                           float32
conf_spots                         float32
Length: 128, dtype: object


### Making final validations:
As the dataset has totally cleaned and many transformations has been applied on the data, it may be good if we made one final validation to ensure that everything is ready.

#### Validating the seasons matches counts:

In [104]:
matches_interim_df.groupby('season').size()

season
2014/2015    380
2015/2016    380
2016/2017    380
2017/2018    380
2018/2019    380
2019/2020    380
2020/2021    380
2021/2022    380
2022/2023    380
2023/2024    380
2024/2025    380
dtype: int64

### Validating the amount of unique teams in each of the seasons:

In [105]:
matches_interim_df.groupby('season')['h_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: h_title, dtype: int64

In [106]:
matches_interim_df.groupby('season')['a_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: a_title, dtype: int64

### Validating amount of matches for each of the teams in each of seasons:
Each team should have played a total of **38** matches per season.To check that, I will use the `validate_team_matches` function which will return the amount of seasons in which all of the teams has played a total of **38** matches!We expect to get **11** valid seasons:

In [107]:
validate_team_matches(matches_interim_df, 'h_title', 'a_title')

{'2014/2015': 20,
 '2015/2016': 20,
 '2016/2017': 20,
 '2017/2018': 20,
 '2018/2019': 20,
 '2019/2020': 20,
 '2020/2021': 20,
 '2021/2022': 20,
 '2022/2023': 20,
 '2023/2024': 20,
 '2024/2025': 20}

### Checking for duplicates:

In [108]:
matches_interim_df.duplicated(
    subset=[
       'date', 'h_title', 'a_title' 
    ]
).any()

np.False_

In [109]:
matches_interim_df.shape

(4180, 128)

Everything seems to be perfect!

### Saving the dataset:
We can finally save the final version of the Transfermarkt data!I will use the `save_data` function and save the data into the transfermarkt/interim folder under the final 3rd version of the data!

In [110]:
PROJECT_ROOT = Path().resolve().parent.parent

file_path = PROJECT_ROOT / TRANSFERMARKT_DATA_INTERIM_PATH
file_path.mkdir(parents=True, exist_ok=True)

In [111]:
file_path = PROJECT_ROOT / TRANSFERMARKT_DATA_INTERIM_PATH / 'interim_matches_v3.parquet'

save_data(data=matches_interim_df, file_path=file_path)

The file has already been created and it contains the exact data as the original dataset!
